[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week6_chest_xray_STARTER.ipynb)


# Week 6 -- Medical Imaging — The Chest X-Ray (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** NIH Chest X-Ray14 — multi-label classification, BCEWithLogitsLoss, AUC-ROC, missing labels

**Dataset:** NIH Chest X-Ray14

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


## Learning Objectives

By the end of this week, you will be able to:

- Frame multi-label classification correctly: each image has multiple simultaneous binary labels
- Replace CrossEntropyLoss with BCEWithLogitsLoss and explain why
- Compute AUC-ROC per pathology label and explain why it differs from F1
- Handle the missing label problem in NIH Chest X-Ray14
- Fine-tune a pretrained model for multi-label classification



---

## MONDAY -- "Multi-Label Classification — The Right Framework"


Single-label: one image, one class. Multi-label: one image, multiple classes simultaneously. A chest X-ray of a patient with pneumonia AND cardiomegaly AND effusion has three labels. Softmax (CrossEntropyLoss) sums to 1 across all classes — impossible for multi-label. Sigmoid (BCEWithLogitsLoss) treats each class independently — each has its own probability in [0,1].


### Exercise 6.1 -- Loss function experiment

Apply CrossEntropyLoss to a synthetic 3-hot target vector (three conditions active simultaneously). Print the gradient. Apply BCEWithLogitsLoss to the same target. Compare: which loss function correctly penalises the model for missing the second and third conditions?


In [ ]:
# Exercise 6.1: Loss function experiment
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: Multi-Label Classification — The Right Framework (scaffold) --
import torch
import torch.nn as nn

# Single-label (WRONG for multi-label)
loss_wrong = nn.CrossEntropyLoss()  # softmax internally — sums to 1

# Multi-label (CORRECT)
loss_correct = nn.BCEWithLogitsLoss()  # sigmoid per label — independent

# Example
logits = torch.randn(4, 14)   # (batch=4, 14 labels)
targets = torch.zeros(4, 14)  # binary: 0 or 1 per label
targets[0, [0, 3, 7]] = 1     # patient 0 has conditions 0, 3, and 7
loss = loss_correct(logits, targets)
print(f"Multi-label loss: {loss.item():.4f}")


### Monday Morning Moment

*Slack — Monday, 10:00am.*

**Ngozi Eze-Okafor:** Quick question — the loss function. The dataset has 14 labels. I was using CrossEntropyLoss. Yewande says to switch.

**Yewande Adeyemi:** BCEWithLogitsLoss. Each label is independent.

**Dr. Kwame Asante:** Ngozi, run this experiment before switching: apply CrossEntropyLoss to a patient with conditions [Pneumonia=1, Effusion=1, Cardiomegaly=1]. What does the softmax output look like?

**Ngozi Eze-Okafor:** The probabilities sum to 1 across all 14 conditions. So if Pneumonia gets 0.4, the remaining 0.6 is distributed across the other 13 labels — including the ones that are also present.

**Dr. Kwame Asante:** So CrossEntropyLoss forces the model to choose one condition as the most likely. For a patient who has three simultaneous conditions — 

**Ngozi Eze-Okafor:** It will underestimate all three. I understand now.




---

## TUESDAY -- "AUC-ROC — The Right Metric for Clinical AI"


F1 requires choosing a threshold (usually 0.5). AUC-ROC measures discriminative ability across all possible thresholds. For a rare condition (1% prevalence), F1 at threshold 0.5 can be near-zero even for a useful model. AUC-ROC remains interpretable: 0.5 = random, 1.0 = perfect. For clinical AI, report AUC-ROC per label, not aggregate F1.


### Exercise 6.2 -- AUC vs F1 comparison

For the Hernia condition (rarest in the dataset), compute both F1 (threshold=0.5) and AUC-ROC. Why is F1 so low even when the model has some discriminative ability? At what threshold does F1 maximise?


In [ ]:
# Exercise 6.2: AUC vs F1 comparison
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: AUC-ROC — The Right Metric for Clinical AI (scaffold) --
from sklearn.metrics import roc_auc_score
import numpy as np

# Per-label AUC-ROC
CONDITIONS = ["Atelectasis","Cardiomegaly","Effusion","Infiltration",
              "Mass","Nodule","Pneumonia","Pneumothorax","Consolidation",
              "Edema","Emphysema","Fibrosis","Pleural_Thickening","Hernia"]

# y_true: (N, 14) binary labels
# y_score: (N, 14) predicted probabilities (after sigmoid)
for i, cond in enumerate(CONDITIONS):
    if y_true[:,i].sum() > 0:  # skip if no positive examples
        auc = roc_auc_score(y_true[:,i], y_score[:,i])
        print(f"  {cond:<25}: AUC = {auc:.4f}")



---

## WEDNESDAY -- "Missing Labels — The Elephant in the Room"


In NIH Chest X-Ray14, labels were extracted from radiology reports using NLP. If a radiologist did not mention cardiomegaly in a report, the label is 0 — even if cardiomegaly is present but not the focus of the report. Approximately 10–15% of labels may be systematically missing. Training with BCEWithLogitsLoss treats these zeros as confirmed negatives, which they are not.


### Exercise 6.3 -- Missing label audit

For each of the 14 conditions, compute: (a) positive label rate, (b) co-occurrence rate with other conditions, (c) estimated missing label rate (from literature: ~12% for NIH CXR14). Plot missing label rate vs AUC. Is there a correlation?


In [ ]:
# Exercise 6.3: Missing label audit
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: Missing Labels — The Elephant in the Room (scaffold) --
# The missing label problem
# Label = 0 can mean:
# (a) Condition is truly absent — true negative
# (b) Condition was not mentioned in the report — unknown
# (c) Condition was outside the radiologist's focus — unknown

# Partial mitigation: use label smoothing
loss_smooth = nn.BCEWithLogitsLoss(reduction="mean")
# Or: use partial labels — treat 0 as unlabelled, not negative
# (requires a different loss formulation)



---

## THURSDAY -- "Fine-Tuning ResNet-18 for Multi-Label"


Replace the final fully-connected layer with a new one that has 14 outputs instead of 1000. Freeze the backbone for 2 epochs (only the new head trains). Then unfreeze and fine-tune the entire network at a lower learning rate.


### STOP AND TRACE

Before running: what does sigmoid(0.0) equal? What does sigmoid(3.0) equal?

loss = nn.BCEWithLogitsLoss()
logit = torch.tensor([0.0])  # model output before sigmoid
target = torch.tensor([1.0])  # true label: condition present
result = loss(logit, target)

Compute by hand: -[1 × log(sigmoid(0)) + 0 × log(1-sigmoid(0))]
Why this line exists: BCEWithLogitsLoss combines sigmoid + binary cross-entropy in one numerically stable operation. Computing sigmoid then BCE separately can produce NaN for large logit values.


In [ ]:
# Exercise 6.4: STOP AND TRACE — BCEWithLogitsLoss
# -------------------------------------------------------
# STOP AND TRACE: before running, write your expected output as a comment.
# Then run the cell and compare.

# YOUR PREDICTION:
# Expected output: ???

# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: Fine-Tuning ResNet-18 for Multi-Label (scaffold) --
import torchvision.models as models

model = models.resnet18(weights="IMAGENET1K_V1")
# Replace the classifier head for 14-label output
model.fc = nn.Linear(model.fc.in_features, 14)  # 14 conditions

# Phase 1: freeze backbone, train head only
for name, param in model.named_parameters():
    if "fc" not in name: param.requires_grad = False
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

# Phase 2 (after 2 epochs): unfreeze all
for param in model.parameters(): param.requires_grad = True
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # lower LR for full network



---

## FRIDAY -- "The Friday Build: Clinical AUC Report"


Report per-label AUC-ROC for all 14 conditions. Translate the three worst-performing labels into clinical sentences. For each: what does low AUC mean for a patient? Which conditions have the most missing labels? Is there a correlation between missing label rate and AUC?


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Friday Build: Clinical AUC Report (scaffold) --
# Clinical AUC report structure:
# 1. Per-label AUC table (all 14 conditions)
# 2. The three worst: clinical translation
# 3. Missing label rate per condition
# 4. Correlation: missing labels vs AUC
# 5. One recommendation for improving the lowest-AUC condition


### Friday Workplace Moment

*MediVision AI — Friday standup.*

**Ngozi Eze-Okafor:** Hernia has AUC 0.71. Pneumonia has AUC 0.68. Both are below 0.75 — the threshold we set for clinical use.

**Dr. Kwame Asante:** What is the missing label rate for Hernia?

**Ngozi Eze-Okafor:** 23% of Hernia-positive images have no other pathology labelled. I suspect the NLP label extraction missed hernia mentions that were framed as incidental findings.

**Nurse Folake Balogun:** Hernia is often incidental. A radiologist reading a pneumonia film might not mention a small hiatal hernia.

**Dr. Kwame Asante:** So the model's AUC for Hernia is bounded above by the label quality. The model cannot outperform its training labels.

**Ngozi Eze-Okafor:** Which means the 0.68 might be the model working correctly on broken labels, not a model failure.

**Dr. Kwame Asante:** Write that in the evaluation report under "Limitations".



## YOUR WEEK 6 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] I can explain why CrossEntropyLoss is wrong for multi-label classification — with a concrete example.
- [ ] I can compute AUC-ROC for one condition by hand and explain why it is more useful than F1 for rare conditions.
- [ ] I know the missing label problem in NIH Chest X-Ray14 and can state its estimated rate from memory.
- [ ] My ResNet-18 fine-tuning used two-phase training: frozen backbone first, then full network.
- [ ] My clinical AUC report translates the three worst-performing conditions into clinical sentences.


## Challenge Task

> **Core path:** Implement class-balanced sampling for multi-label data: for each batch, ensure each condition is represented at least once. Compare training loss curves with and without balanced sampling.

> **Advanced path:** Implement label smoothing (replace 0→ε, 1→1-ε) and test its effect on AUC for the three conditions with the highest missing label rates. Does smoothing help?


## Common Mistakes

**Using CrossEntropyLoss for multi-label problems:** CrossEntropyLoss applies softmax — probabilities sum to 1. Multi-label targets can have multiple 1s simultaneously. Use BCEWithLogitsLoss.

**Treating missing labels as confirmed negatives:** In NIH Chest X-Ray14, label=0 means "not mentioned in the report", not "confirmed absent". Training with zeros-as-negatives introduces systematic bias.

**Reporting aggregate AUC only:** Mean AUC across 14 conditions hides that Hernia (AUC=0.71) and Pneumonia (AUC=0.68) are below clinical threshold while Cardiomegaly (AUC=0.92) is not.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
